# Analyse du Chiffre d'Affaires — `resultats_final.csv`
> Visualisations interactives avec **Plotly**.

In [ ]:
# !pip install plotly pandas openpyxl

In [ ]:
ca_total   = df['CA_Total'].sum()
tva_tot    = df['TVA'].sum()
remise_tot = df['remise_eur'].sum()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Top 10 CA Total', 'Distribution des prix unitaires',
        'Distribution des remises (%)', 'Distribution du CA Net'
    ),
    specs=[
        [{'type': 'pie'}, {'type': 'xy'}],
        [{'type': 'xy'}, {'type': 'xy'}]
    ],
    vertical_spacing=0.20, horizontal_spacing=0.14
)
top10 = df.nlargest(10, 'CA_Total')
fig.add_trace(go.Pie(labels=top10['ID'], values=top10['CA_Total'],
    hole=0.4, textinfo='label+percent', textfont=dict(size=10), showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=df['Prix'], nbinsx=20, marker_color='#2563eb', opacity=0.8, name='Prix'), row=1, col=2)
fig.add_trace(go.Histogram(x=df['Remise'], nbinsx=12, marker_color='#f59e0b', opacity=0.8, name='Remise'), row=2, col=1)
fig.add_trace(go.Histogram(x=df['CA_Net'], nbinsx=20, marker_color='#10b981', opacity=0.8, name='CA Net'), row=2, col=2)
fig.update_layout(
    title=dict(text=f'Dashboard — CA Total : {ca_total:,.0f} € | TVA : {tva_tot:,.0f} € | Remises : {remise_tot:,.0f} €', font_size=15),
    template=TEMPLATE, height=800, width=1200, showlegend=False
)
fig.show()


In [ ]:
df = pd.read_csv(r'c:\Matière logiciels\VS code\projet PFA\ventes_clean.csv')

# Calcul des indicateurs
df['CA_Brut']    = (df['Prix'] * df['Quantite']).round(2)
df['CA_Net']     = (df['CA_Brut'] * (1 - df['Remise'] / 100)).round(2)
df['TVA']        = (df['CA_Net'] * 0.20).round(2)
df['CA_Total']   = (df['CA_Net'] + df['TVA']).round(2)
df['remise_eur'] = (df['CA_Brut'] - df['CA_Net']).round(2)
df['taux_marge'] = (df['TVA'] / df['CA_Net'] * 100).round(1)
df['ID'] = df['ID'].astype(str)

print(f'{len(df)} produits chargés')
df.head()


---
## 1 · CA Total par produit

In [ ]:
df_top30 = df.sort_values('CA_Total', ascending=False).head(30)

fig = px.bar(
    df_top30,
    x='ID', y='CA_Total',
    color='CA_Total',
    color_continuous_scale='Blues',
    labels={'ID': 'ID Produit', 'CA_Total': 'CA Total (€)'},
    title='Top 30 produits — CA Total (€)',
    template=TEMPLATE
)
fig.update_traces(marker_line_width=0, hovertemplate='<b>Produit %{x}</b><br>CA Total : %{y:,.0f} €<extra></extra>')
fig.update_layout(
    coloraxis_showscale=False, title_font_size=18,
    height=500, width=1100,
    xaxis=dict(tickangle=-45, tickfont=dict(size=9), title='ID Produit'),
    yaxis=dict(gridcolor='#f0f0f0', title='CA Total (€)'),
    margin=dict(b=100)
)
fig.show()


---
## 2 · CA Brut vs CA Net — impact des remises

In [ ]:
df_rem = df.sort_values('remise_eur', ascending=False).head(30)

fig = go.Figure()
fig.add_trace(go.Bar(
    name='CA Net', x=df_rem['ID'], y=df_rem['CA_Net'],
    marker_color='#2563eb',
    hovertemplate='<b>Produit %{x}</b><br>CA net : %{y:,.0f} €<extra></extra>'
))
fig.add_trace(go.Bar(
    name='Remise appliquée', x=df_rem['ID'], y=df_rem['remise_eur'],
    marker_color=ACCENT, opacity=0.75,
    hovertemplate='<b>Produit %{x}</b><br>Remise : %{y:,.0f} €<extra></extra>'
))
fig.update_layout(
    barmode='stack',
    title='Top 30 produits remisés — CA Net + Remises',
    title_font_size=18,
    xaxis=dict(tickangle=-45, tickfont=dict(size=9), title='ID Produit'),
    yaxis=dict(gridcolor='#f0f0f0', title='CA (€)'),
    template=TEMPLATE, height=500, width=1100,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(b=100)
)
fig.show()


---
## 3 · TVA par produit

In [ ]:
df_t = df.sort_values('TVA', ascending=True)

fig = go.Figure(go.Bar(
    x=df_t['TVA'], y=df_t['ID'],
    orientation='h',
    marker_color='#2563eb',
    hovertemplate='<b>Produit %{y}</b><br>TVA : %{x:,.0f} €<extra></extra>'
))
fig.update_layout(
    title='TVA collectée par produit',
    title_font_size=18,
    xaxis=dict(gridcolor='#f0f0f0', title='TVA (€)'),
    yaxis=dict(title='ID Produit', tickfont=dict(size=7)),
    template=TEMPLATE,
    height=max(800, len(df) * 12),
    width=900,
    margin=dict(l=70)
)
fig.show()


---
## 4 · Scatter : CA Net vs TVA (bulle = quantité, couleur = remise)

In [ ]:
fig = px.scatter(
    df,
    x='CA_Net', y='TVA',
    size='Quantite', color='Remise',
    color_continuous_scale='RdYlGn_r',
    hover_name='ID',
    hover_data={'CA_Net': ':,.0f', 'TVA': ':,.0f', 'Quantite': True, 'Remise': True},
    labels={'CA_Net': 'CA Net (€)', 'TVA': 'TVA (€)', 'Remise': 'Remise (%)', 'Quantite': 'Quantité'},
    title='Relation TVA / CA Net — taille = quantité, couleur = remise',
    template=TEMPLATE
)
fig.update_traces(marker_line_width=0.5, marker_line_color='white', marker=dict(sizemin=4))
fig.update_layout(title_font_size=18, height=600, width=1000,
    xaxis=dict(gridcolor='#f0f0f0'), yaxis=dict(gridcolor='#f0f0f0'))
fig.show()


---
## 5 · Dashboard récapitulatif

In [ ]:
ca_total   = df['CA_Total'].sum()
tva_tot    = df['TVA'].sum()
remise_tot = df['remise_eur'].sum()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Top 10 CA Total', 'Distribution des prix unitaires',
        'Distribution des remises (%)', 'Distribution du CA Net'
    ),
    specs=[
        [{'type': 'pie'}, {'type': 'xy'}],
        [{'type': 'xy'}, {'type': 'xy'}]
    ],
    vertical_spacing=0.20, horizontal_spacing=0.14
)
top10 = df.nlargest(10, 'CA_Total')
fig.add_trace(go.Pie(labels=top10['ID'], values=top10['CA_Total'],
    hole=0.4, textinfo='label+percent', textfont=dict(size=10), showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=df['Prix'], nbinsx=20, marker_color='#2563eb', opacity=0.8, name='Prix'), row=1, col=2)
fig.add_trace(go.Histogram(x=df['Remise'], nbinsx=12, marker_color='#f59e0b', opacity=0.8, name='Remise'), row=2, col=1)
fig.add_trace(go.Histogram(x=df['CA_Net'], nbinsx=20, marker_color='#10b981', opacity=0.8, name='CA Net'), row=2, col=2)
fig.update_layout(
    title=dict(text=f'Dashboard — CA Total : {ca_total:,.0f} € | TVA : {tva_tot:,.0f} € | Remises : {remise_tot:,.0f} €', font_size=15),
    template=TEMPLATE, height=800, width=1200, showlegend=False
)
fig.show()


---
## 6 · Export (optionnel)

In [ ]:
fig.write_html(r'c:\Matière logiciels\VS code\projet PFA\dashboard_ca.html')
print('Export terminé !')
